In [1]:
import pandas as pd
df = pd.read_csv('author_ru_utf.csv', sep = ',', encoding = 'utf8')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'author_ru_utf.csv'

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Activation

import numpy as np

def get_model():
    # function returns the many-layered feedforward network
    # considers 9 floating point numbers as input
    # produces 2 floating point numbers as output
    model = Sequential()
    model.add(Dense(18, activation='relu', input_dim=9))
    # uncomment the following line to add one more layer
    # model.add(Dense(9, activation='relu'))
    model.add(Dense(2, activation='softmax'))
    model.compile(optimizer='adam',
                  loss='mean_squared_error',
                  metrics=['accuracy'])
    return model

def prepare_data(filename):
    # function reads data from @filename
    # and returns six arrays: train data, train labels,
    # validation data, validation labels, test data and test labels
    with open(filename) as f:
        lines = f.readlines()

    input_data = np.ndarray(shape=(len(lines) - 7, 9), dtype=float)
    output_data = np.ndarray(shape=(len(lines) - 7, 2), dtype=float)

    # skips first 7 lines of text
    for i in range(7, len(lines)):
        line = lines[i].rstrip()
        line_data = line.split(' ')
        ind = i - 7
        input_data[ind, 0:9] = line_data[0:9]
        output_data[ind, 0:2] = line_data[9:11]

    train_data = np.ndarray(shape=(350, 9), dtype=float)
    train_labels = np.ndarray(shape=(350, 2), dtype=float)
    train_data[:, :] = input_data[0:350, :]
    train_labels[:, :] = output_data[0:350, :]

    valid_data = np.ndarray(shape=(175, 9), dtype=float)
    valid_labels = np.ndarray(shape=(175, 2), dtype=float)
    valid_data[:, :] = input_data[350:525, :]
    valid_labels[:, :] = output_data[350:525, :]

    test_data = np.ndarray(shape=(174, 9), dtype=float)
    test_labels = np.ndarray(shape=(174, 2), dtype=float)
    test_data[:, :] = input_data[525:699, :]
    test_labels[:, :] = output_data[525:699, :]

    return train_data, train_labels, valid_data, valid_labels, test_data, test_labels

In [ ]:
# getting data for both training and testing
train_data, train_labels, valid_data, valid_labels, test_data, test_labels = prepare_data('cancer1.dt')

# creating a model
model = get_model()

# training a model
model.fit(train_data, train_labels, validation_data=(valid_data, valid_labels),
          epochs=100, batch_size=32)

# computing network predictions
predictions = model.predict_on_batch(test_data)

# computing the percentage of correct predictions
res = 0.0
for i in range(test_data.shape[0]):
    r1 = 0
    if (test_labels[i, 0] < test_labels[i, 1]):
        r1 = 1
    r2 = 0
    if (predictions[i, 0] < predictions[i, 1]):
        r2 = 1
    if (r1 == r2):
        res += 1.0
res /= float(test_data.shape[0])

# printing resulting percentage
print('-------------------')
print(res * 100.0)
print('-------------------')
